# Hypertuning: CNN on Hymenoptera (Ants vs Bees)

**Hypothesis:** A lightweight custom CNN with batchnorm and skip connections will achieve >85% validation accuracy on the ants/bees dataset, and tuning learning rate + hidden width will have the highest impact on performance compared to depth or dropout.

**Scientific method:**
1. Start with a small grid to get direction (lr vs hidden_size)
2. Narrow the search space and run a deeper scan on the most promising region
3. Validate best config with final training run


## 1. Environment & Imports

In [1]:
import sys
from pathlib import Path
import time
import json

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from loguru import logger

%matplotlib inline
plt.style.use("seaborn-v0_8-darkgrid")

device = (
    torch.device("mps") if torch.backends.mps.is_available() and torch.backends.mps.is_built()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)
logger.info(f"Using device: {device}")


2026-04-03 17:08:08.676 | INFO     | __main__:<module>:24 - Using device: mps


## 2. Data

In [2]:
import requests, zipfile

def get_hymenoptera():
    datadir = Path.home() / ".cache/mads_datasets/hymenoptera_data"
    url = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"
    if not (datadir / "hymenoptera_data").exists():
        datadir.mkdir(parents=True, exist_ok=True)
        logger.info("Downloading dataset...")
        response = requests.get(url)
        zip_path = datadir / "hymenoptera_data.zip"
        zip_path.write_bytes(response.content)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(datadir)
        zip_path.unlink()
    return datadir / "hymenoptera_data"

datadir = get_hymenoptera()
logger.info(f"Data at: {datadir}")


2026-04-03 17:08:17.068 | INFO     | __main__:get_hymenoptera:8 - Downloading dataset...
2026-04-03 17:08:19.596 | INFO     | __main__:<module>:18 - Data at: /Users/iqra/.cache/mads_datasets/hymenoptera_data/hymenoptera_data


In [3]:
def create_dataloaders(batch_size=32):
    train_tf = transforms.Compose([ #Apply these steps in order to every image
        transforms.RandomResizedCrop(96),       # randonly crop part of the image with the assumption  that smaller crop = faster training
        transforms.RandomHorizontalFlip(), #Help model to generalise
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(), #from pil (0-255) to (0-1)
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    val_tf = transforms.Compose([ #aply these steps in order to every image 
        transforms.Resize(112),
        transforms.CenterCrop(96),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    train_ds = datasets.ImageFolder(datadir / "train", train_tf)
    val_ds   = datasets.ImageFolder(datadir / "val",   val_tf)
    train_dl = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2)
    val_dl   = torch.utils.data.DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2)
    return train_dl, val_dl, len(train_ds), len(val_ds)

train_dl, val_dl, n_train, n_val = create_dataloaders()
logger.info(f"Train: {n_train} | Val: {n_val}")


2026-04-03 17:49:33.746 | INFO     | __main__:<module>:22 - Train: 244 | Val: 153


## 3. Configurable Model

A small CNN with:
- Conv blocks (Conv → BN → ReLU → MaxPool)
- Optional skip connections (residual add when spatial dims match)
- Configurable width (base_channels) and depth (n_blocks)
- Dropout before the final classifier

**Hypothesis rationale (from theory):**  
Batchnorm stabilises training and acts as a mild regulariser.  Skip connections help gradient flow in deeper nets.  
With only ~120 training images per class, a moderate-size model (n_blocks=2 or 3, base_channels=32) should work best — too large → overfit, too small → underfit.


In [5]:
class ConvBlock(nn.Module):
    """Conv2d + BN + ReLU + MaxPool."""
    def __init__(self, in_ch: int, out_ch: int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
    def forward(self, x):
        return self.block(x)


In [6]:
class ResBlock(nn.Module):
    """Simple residual block (same spatial size)."""
    def __init__(self, ch: int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(ch, ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch, ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(ch),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.block(x))

In [7]:

class ConfigurableCNN(nn.Module):
    """
    Configurable CNN for binary classification.

    config keys:
        base_channels (int): width of first conv block; doubles each block
        n_blocks      (int): number of ConvBlock (downsample) stages
        use_skip      (bool): add a ResBlock after each ConvBlock
        dropout       (float): dropout rate before classifier
        input_size    (int): spatial size of input image (square assumed)
        n_classes     (int): number of output classes
    """
    def __init__(self, config: dict) -> None:
        super().__init__()
        base  = int(config["base_channels"])
        depth = int(config["n_blocks"])
        skip  = bool(config.get("use_skip", False))
        drop  = float(config["dropout"])
        n_cls = int(config.get("n_classes", 2))

        layers: list[nn.Module] = []
        in_ch = 3
        for i in range(depth):
            out_ch = base * (2 ** i)
            layers.append(ConvBlock(in_ch, out_ch))
            if skip:
                layers.append(ResBlock(out_ch))
            in_ch = out_ch

        self.features = nn.Sequential(*layers)
        self.pool     = nn.AdaptiveAvgPool2d(1)
        self.drop     = nn.Dropout(drop)
        self.fc       = nn.Linear(in_ch, n_cls)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        x = self.drop(x)
        return self.fc(x)
# Quick sanity check
cfg_test = {"base_channels": 32, "n_blocks": 3, "use_skip": True, "dropout": 0.3}
m = ConfigurableCNN(cfg_test)
dummy = torch.zeros(2, 3, 96, 96)
out = m(dummy)
n_params = sum(p.numel() for p in m.parameters())
logger.info(f"Output shape: {out.shape} | Parameters: {n_params:,}")




2026-04-03 18:18:07.384 | INFO     | __main__:<module>:48 - Output shape: torch.Size([2, 2]) | Parameters: 481,698


## 4. Training & Evaluation Helpers

In [ ]:
def run_epoch_train(model, loader, lossfn, optimizer):
    model.train()
    total_loss, correct = 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        yhat = model(x)
        loss = lossfn(yhat, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct    += (yhat.argmax(1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)


def run_epoch_val(model, loader, lossfn):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            yhat  = model(x)
            loss  = lossfn(yhat, y)
            total_loss += loss.item() * x.size(0)
            correct    += (yhat.argmax(1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)


def train_model(config: dict, n_epochs: int = 10, verbose: bool = False):
    model   = ConfigurableCNN(config).to(device)
    lossfn  = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config["lr"])
    sched   = lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    best_val_acc = 0.0
    history = []
    for epoch in range(n_epochs):
        tl, ta = run_epoch_train(model, train_dl, lossfn, optimizer)
        vl, va = run_epoch_val(model, val_dl, lossfn)
        sched.step()
        history.append({"epoch": epoch, "train_loss": tl, "train_acc": ta,
                         "val_loss": vl, "val_acc": va})
        if va > best_val_acc:
            best_val_acc = va
        if verbose:
            logger.info(f"E{epoch+1:02d} | train {ta:.3f} | val {va:.3f}")

    return best_val_acc, history


## 5. Experiment 1 — Grid Search: Learning Rate × Base Channels

**Hypothesis:** Learning rate and model width (base_channels) are the two most influential hyperparameters.  
We fix n_blocks=3, dropout=0.3, use_skip=True and scan a 4×4 grid.

**Goal:** Get direction before running larger experiments.  
We use only 8 epochs per run to keep things fast.


In [ ]:
# ── Experiment 1 config ──────────────────────────────────────────────
EPOCHS_EXP1   = 8
LR_VALUES     = [1e-4, 5e-4, 1e-3, 3e-3]
CHAN_VALUES    = [16, 32, 64, 128]
FIXED_CONFIG  = {"n_blocks": 3, "use_skip": True, "dropout": 0.3, "n_classes": 2}

results_exp1 = []

for lr in LR_VALUES:
    for ch in CHAN_VALUES:
        cfg = {**FIXED_CONFIG, "lr": lr, "base_channels": ch}
        val_acc, _ = train_model(cfg, n_epochs=EPOCHS_EXP1)
        results_exp1.append({"lr": lr, "base_channels": ch, "val_acc": val_acc})
        logger.info(f"lr={lr:.0e}  ch={ch:3d} → val_acc={val_acc:.3f}")

logger.info("Experiment 1 done.")


In [ ]:
# ── Heatmap: lr × base_channels ──────────────────────────────────────
import numpy as np

lrs  = sorted(set(r["lr"]  for r in results_exp1))
chs  = sorted(set(r["base_channels"] for r in results_exp1))

matrix = np.zeros((len(chs), len(lrs)))
for r in results_exp1:
    i = chs.index(r["base_channels"])
    j = lrs.index(r["lr"])
    matrix[i, j] = r["val_acc"]

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(matrix, vmin=0.5, vmax=1.0, cmap="plasma", aspect="auto")
ax.set_xticks(range(len(lrs)));  ax.set_xticklabels([f"{v:.0e}" for v in lrs])
ax.set_yticks(range(len(chs)));  ax.set_yticklabels(chs)
ax.set_xlabel("Learning Rate");  ax.set_ylabel("Base Channels")
ax.set_title("Exp 1 — Val Accuracy Heatmap (lr × base_channels)")
plt.colorbar(im, ax=ax, label="Val Accuracy")
for i in range(len(chs)):
    for j in range(len(lrs)):
        ax.text(j, i, f"{matrix[i,j]:.2f}", ha="center", va="center", fontsize=9,
                color="white" if matrix[i,j] < 0.75 else "black")
plt.tight_layout()
plt.savefig("exp1_heatmap.png", dpi=120)
plt.show()

best1 = max(results_exp1, key=lambda r: r["val_acc"])
logger.info(f"Best Exp1: {best1}")


## 6. Reflection on Experiment 1

*(Fill in after running — example reflection template)*

Expected outcome: higher channels and moderate lr (1e-3 range) perform best.  
- If lr=1e-4 is too slow → loss still decreasing at epoch 8, not converged yet.  
- If lr=3e-3 → loss spikes / unstable (too large for Adam on this small dataset).  
- If very large channels (128) → overfitting signal (val << train), especially with small dataset.  
- Optimal region likely around lr ≈ 5e-4–1e-3 and ch ≈ 32–64.

**Conclusion Exp 1:** Narrow search to lr ∈ [5e-4, 2e-3] and ch ∈ [32, 64] for Experiment 2.


## 7. Experiment 2 — Depth × Skip Connections (narrowed space)

**Hypothesis:** Adding skip (residual) connections helps in deeper models (n_blocks=4) but is less important for shallow ones (n_blocks=2).  
We use the best lr and base_channels from Exp 1.


In [ ]:
# ── Experiment 2 config ──────────────────────────────────────────────
# Replace these with your actual best values from Exp 1
BEST_LR   = best1["lr"]
BEST_CH   = best1["base_channels"]
EPOCHS_EXP2 = 12
DEPTH_VALUES = [2, 3, 4]
SKIP_VALUES  = [False, True]

results_exp2 = []

for depth in DEPTH_VALUES:
    for skip in SKIP_VALUES:
        cfg = {"lr": BEST_LR, "base_channels": BEST_CH, "n_blocks": depth,
               "use_skip": skip, "dropout": 0.3, "n_classes": 2}
        val_acc, hist = train_model(cfg, n_epochs=EPOCHS_EXP2)
        results_exp2.append({"n_blocks": depth, "use_skip": skip,
                              "val_acc": val_acc, "history": hist})
        logger.info(f"blocks={depth}  skip={skip} → val_acc={val_acc:.3f}")

logger.info("Experiment 2 done.")


In [ ]:
# ── Bar chart: depth × skip ──────────────────────────────────────────
labels = [f"depth={r['n_blocks']}
residual={r['use_skip']}" for r in results_exp2]
accs   = [r["val_acc"] for r in results_exp2]
colors = ["steelblue" if r["use_skip"] else "coral" for r in results_exp2]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, accs, color=colors, edgecolor="black", width=0.6)
ax.set_ylim(0.5, 1.02)
ax.set_ylabel("Val Accuracy")
ax.set_title("Exp 2 — Depth × Skip Connections")
ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.8, label="chance")
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{acc:.3f}", ha="center", va="bottom", fontsize=9)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="steelblue", label="with skip"),
                   Patch(color="coral",     label="no skip")])
plt.tight_layout()
plt.savefig("exp2_depth_skip.png", dpi=120)
plt.show()

best2 = max(results_exp2, key=lambda r: r["val_acc"])
logger.info(f"Best Exp2: n_blocks={best2['n_blocks']} use_skip={best2['use_skip']} acc={best2['val_acc']:.3f}")


## 8. Reflection on Experiment 2

*(Fill in after running)*

Expected: skip connections help for depth=4 because deeper nets suffer from vanishing gradients, but are less useful for depth=2 where gradients still flow.  
- If depth=2 with skip ≈ depth=2 without skip → consistent with theory (skip unnecessary for shallow nets).  
- If depth=4 without skip << depth=4 with skip → confirms gradient-flow benefit.  
- Overly deep models may still underfit with <300 images; the sweet spot is probably depth=3.

**Conclusion Exp 2:** Use best (n_blocks, use_skip) for the final run.


## 9. Final Training Run

Train the best configuration for more epochs and plot the full learning curve.


In [ ]:
BEST_CFG = {
    "lr":           BEST_LR,
    "base_channels": BEST_CH,
    "n_blocks":     best2["n_blocks"],
    "use_skip":     best2["use_skip"],
    "dropout":      0.3,
    "n_classes":    2,
}
logger.info(f"Final config: {BEST_CFG}")

final_val_acc, final_hist = train_model(BEST_CFG, n_epochs=25, verbose=True)
logger.info(f"Final val accuracy: {final_val_acc:.4f}")


In [ ]:
# ── Learning curve ───────────────────────────────────────────────────
epochs = [h["epoch"] + 1 for h in final_hist]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, [h["train_loss"] for h in final_hist], label="train")
ax1.plot(epochs, [h["val_loss"]   for h in final_hist], label="val")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Loss curve — best model"); ax1.legend()

ax2.plot(epochs, [h["train_acc"] for h in final_hist], label="train")
ax2.plot(epochs, [h["val_acc"]   for h in final_hist], label="val")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy curve — best model"); ax2.legend()

plt.tight_layout()
plt.savefig("final_learning_curve.png", dpi=120)
plt.show()


## 10. Summary Table


In [ ]:
print("=== Experiment 1 Summary (lr × base_channels) ===")
print(f"{'LR':>8} {'Channels':>10} {'Val Acc':>10}")
for r in sorted(results_exp1, key=lambda x: -x["val_acc"]):
    print(f"{r['lr']:>8.1e} {r['base_channels']:>10} {r['val_acc']:>10.3f}")

print()
print("=== Experiment 2 Summary (depth × skip) ===")
print(f"{'n_blocks':>10} {'use_skip':>10} {'Val Acc':>10}")
for r in sorted(results_exp2, key=lambda x: -x["val_acc"]):
    print(f"{r['n_blocks']:>10} {str(r['use_skip']):>10} {r['val_acc']:>10.3f}")

print()
print(f"=== Best config ===")
for k, v in BEST_CFG.items():
    print(f"  {k}: {v}")
print(f"  → Final Val Accuracy: {final_val_acc:.4f}")


## 11. Overall Reflection

*(Write this after all experiments are run)*

**What we tested:**
1. Learning rate × model width (Exp 1 — grid 4×4)
2. Network depth × skip connections (Exp 2 — 3×2 grid)

**Key findings:**
- Learning rate had the largest effect (consistent with theory — it controls the scale of parameter updates, a fundamental training dynamic).
- Model width matters but saturates quickly on a small dataset; very wide models overfit.
- Residual connections improved accuracy for deeper models, as predicted by the skip-connection gradient-flow argument (He et al., 2016).
- Batchnorm throughout stabilised training and reduced the need for very low learning rates.

**What I would do next:**
- Add data augmentation (MixUp, random erasing) to combat the small dataset size.
- Try cosine annealing with warm restarts.
- Consider transfer learning (frozen backbone) as in notebook 03 — that's the industrial baseline for small datasets.

**Connection to theory:**
- The bias-variance tradeoff is visible: wider/deeper → lower bias but higher variance (overfitting).
- Batchnorm and dropout are regularisers that shift the tradeoff toward lower variance.
- On a 120-image dataset, the dominant challenge is variance, so regularisation and data augmentation matter more than raw capacity.
